# 🌊 AquaGuard AI — Day 10: Environmental EDA & Cross-Sensor Correlations
- **Date:** 29 Aug 2026
- **Phase:** Phase 1: Ingestion & Governance
- **Objective:** Explore physical trends, distributions, and cross-sensor correlations on the training partition only.
- **Inputs Consumed:** `data/raw/underwater_sensor_stream.csv`, `artifacts/day09/split.json`
- **Outputs Produced:** `artifacts/day10/eda_correlation_matrix.png`, `artifacts/day10/eda_sensor_distributions.png`, `artifacts/day10/hypotheses_table.csv`
- **Deliverable Satisfied:** Comprehensive EDA visuals and physical environmental hypotheses table.

In [6]:
import os, sys
from pathlib import Path
# Universal project root detection (works from any clone directory or depth)
def _find_project_root():
    curr = Path.cwd()
    for p in [curr] + list(curr.parents):
        if (p / "pyproject.toml").exists() or (p / "requirements.txt").exists():
            return p
    return curr

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

import pandas as pd
from src.utils import set_seed, load_config, ensure_dir
from src.io import load_raw_data, load_artifact
from src.plotting import plot_sensor_streams, plot_correlation_heatmap, plot_distributions

set_seed(42)
config = load_config("configs/config.yaml")
ensure_dir("artifacts/day10")

# Load Day 09 split indices to isolate Train partition
split_indices = load_artifact("artifacts/day09/split.json", required_by_day="Day 09")
df_raw = load_raw_data(config["paths"]["raw_temporal_data"])
train_df = df_raw.iloc[split_indices["train_indices"]].copy()
print(f"Leakage Guard: Loaded Train partition only ({len(train_df)} rows)")

2026-10-19 17:37:58 | INFO    | IO - Loaded raw data from data\raw\underwater_sensor_stream.csv: 720 rows, 7 columns


Leakage Guard: Loaded Train partition only (503 rows)


## 1. Multi-Sensor Time Series Trajectories
Visualize continuous sensor dynamics over time on the training set.

In [7]:
sensor_cols = config["track_a"]["sensor_columns"]
ts_col = config["track_a"]["timestamp_column"]

plot_sensor_streams(
    train_df,
    sensors=sensor_cols,
    timestamp_col=ts_col,
    title="Training Partition: Sensor Telemetry Dynamics",
    save_path="artifacts/day10/sensor_streams.png"
)
print("Sensor stream time-series saved to artifacts/day10/sensor_streams.png")

2026-10-19 17:38:03 | INFO    | Plotting - Saved figure to artifacts\day10\sensor_streams.png


Sensor stream time-series saved to artifacts/day10/sensor_streams.png


## 2. Cross-Sensor Correlation Matrix
Compute inter-sensor correlation matrix to uncover physical relationships (e.g. Temperature vs Dissolved Oxygen).

In [8]:
plot_correlation_heatmap(
    train_df,
    cols=sensor_cols,
    title="Train Partition: Sensor Cross-Correlation Heatmap",
    save_path="artifacts/day10/eda_correlation_matrix.png"
)
print("Correlation heatmap saved to artifacts/day10/eda_correlation_matrix.png")

2026-10-19 17:38:07 | INFO    | Plotting - Saved figure to artifacts\day10\eda_correlation_matrix.png


Correlation heatmap saved to artifacts/day10/eda_correlation_matrix.png


## 3. Sensor Density Distributions
Assess skewness, multi-modality, and standard operating envelopes.

In [9]:
plot_distributions(
    train_df,
    cols=sensor_cols,
    title="Train Partition: Sensor Empirical Distributions",
    save_path="artifacts/day10/eda_sensor_distributions.png"
)
print("Sensor distributions saved to artifacts/day10/eda_sensor_distributions.png")

2026-10-19 17:38:14 | INFO    | Plotting - Saved figure to artifacts\day10\eda_sensor_distributions.png


Sensor distributions saved to artifacts/day10/eda_sensor_distributions.png


## 4. Formulate Physical Feature Hypotheses
Log domain hypotheses based on empirical observations to guide Day 13 feature engineering.

In [10]:
hypotheses = pd.DataFrame([
    {
        "hypothesis_id": "H1",
        "description": "Dissolved Oxygen exhibits strong negative correlation with Temperature due to gas solubility physics.",
        "proposed_feature": "temp_x_do interaction & ratio",
        "expected_impact": "Improves DO forecast accuracy at transition hours"
    },
    {
        "hypothesis_id": "H2",
        "description": "Sensors exhibit diurnal 24-hour cycles driven by solar heating and biological respiration.",
        "proposed_feature": "sin_hour, cos_hour cyclic encodings",
        "expected_impact": "Captures cyclical daily drift without trend overfitting"
    },
    {
        "hypothesis_id": "H3",
        "description": "Short-term momentum (1h, 3h lags) provides high predictive power for rapid sensor shifts.",
        "proposed_feature": "Autoregressive lags (1, 2, 3, 6 steps)",
        "expected_impact": "Dominant feature importance in forecaster"
    }
])

hypotheses.to_csv("artifacts/day10/hypotheses_table.csv", index=False)
print("Hypotheses table saved to artifacts/day10/hypotheses_table.csv:")
hypotheses

Hypotheses table saved to artifacts/day10/hypotheses_table.csv:


,hypothesis_id,description,proposed_feature,expected_impact
0,H1,Dissolved Oxygen exhibits strong negative corr...,temp_x_do interaction & ratio,Improves DO forecast accuracy at transition hours
1,H2,Sensors exhibit diurnal 24-hour cycles driven ...,"sin_hour, cos_hour cyclic encodings",Captures cyclical daily drift without trend ov...
2,H3,"Short-term momentum (1h, 3h lags) provides hig...","Autoregressive lags (1, 2, 3, 6 steps)",Dominant feature importance in forecaster


## 📌 Day 10 Execution Findings & Sign-Off
- **Key Finding:** Strong thermal-DO inverse coupling observed. Clear 24h periodicity present.
- **Artifacts Saved:** `artifacts/day10/eda_correlation_matrix.png`, `artifacts/day10/hypotheses_table.csv`.
- **Handoff:** Ready for Day 11 preprocessing pipeline composition and dataset v1 creation.